# 02 — Stage-2 Progressive 512px -> 768px ResViT-Pro Training (Google Colab TPU v2-8 / v5e-1)
Trains `ResViTPro` on **Google Colab TPU (`torch_xla`)** using native `bfloat16` precision across the Two-Phase Curriculum:
- **Phase A (Epochs 1–15, static 512x512):** CORAL Ordinal Focal Loss + Binary Referable-DR BCE.
- **Phase B (Epochs 16–35, static 768x768):** Unlocks learnable GeM Pooling ($p$-norm), Stride-4 FPN micro-lesion features, and ramps up Differentiable `SoftQWKLoss`.
- **TPU Static Graph Rule:** Resolution changes happen **only once** at the Phase A $\rightarrow$ Phase B transition so XLA compiles the graph only twice across the entire 35-epoch run.

In [ ]:
import torch
import torch.nn.functional as F
from src.tier3_grading import ResViTPro
from src.losses import CORALOrdinalFocalLoss, SoftQWKLoss
from src.data.samplers import CurriculumResolutionScheduler
from src.utils.tpu_accelerator import ColabTPUAccelerator

accel = ColabTPUAccelerator(prefer_tpu=True)
print(f'Active Accelerator: {accel.info.device_name} | Precision: {accel.info.amp_dtype}')

scheduler = CurriculumResolutionScheduler(phase_a_epochs=15, phase_b_epochs=20)
model = ResViTPro(fpn_dim=64, num_classes=5, num_heads=4, gem_p=3.0).to(accel.device)

coral_loss_fn = CORALOrdinalFocalLoss(num_classes=5).to(accel.device)
qwk_loss_fn = SoftQWKLoss(num_classes=5).to(accel.device)

for epoch in [1, 15, 16, 35]:
    cfg = scheduler.get_epoch_config(epoch)
    print(cfg)

In [ ]:
# Single Tri-Head Optimization Step on TPU (bfloat16 + xm.optimizer_step)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
x = torch.randn(4, 3, 256, 256, device=accel.device)
y_grades = torch.tensor([0, 1, 2, 4], dtype=torch.long, device=accel.device)
y_referable = (y_grades >= 2).float().unsqueeze(1)

optimizer.zero_grad()
with accel.autocast_context(enabled=True):
    out = model(x)
    tri = out['tri_head']

    loss_ordinal = coral_loss_fn(tri.ordinal_logits, y_grades)
    loss_qwk = qwk_loss_fn(tri.class_probs, y_grades, is_probs=True)
    loss_ref = F.binary_cross_entropy_with_logits(tri.referable_logit, y_referable)
    loss_cont = F.smooth_l1_loss(tri.continuous_score.squeeze(1), y_grades.float())

    total_loss = loss_ordinal + 0.75 * loss_qwk + 0.50 * loss_ref + 0.40 * loss_cont

total_loss.backward()
accel.optimizer_step(optimizer)
print(f'Total Tri-Head Loss: {total_loss.item():.4f} (QWK Loss: {loss_qwk.item():.4f})')
accel.save_checkpoint(model, 'checkpoints/stage2_resvit_pro_768_best.pth')